# ML-10 — Content Action Playbook

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

# Ranked Actions and Reason Codes

The purpose of this playbook is to prioritize content pages for review.

The output is a ranked queue rather than an automatic decision system.

Higher-ranked pages should receive review before lower-ranked pages.

Each recommendation includes a reason code explaining why it appeared in the queue.

In [1]:
#creating Queue
import pandas as pd
import numpy as np

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")

In [2]:
#Creatign Score
df["action_score"] = (
    0.40 * (
        df["impressions_90d"] /
        df["impressions_90d"].max()
    )
    +
    0.30 * (
        df["content_age_days"] /
        df["content_age_days"].max()
    )
    +
    0.30 * (
        1 -
        (
            df["ctr"] /
            max(df["ctr"].max(),0.01)
        )
    )
) * 100

In [3]:
#reason Codes

df["reason_code"] = np.where(
    (
        df["content_age_days"] >= 180
    )
    &
    (
        df["impressions_90d"] >= 500
    ),
    "stale_visible_page",
    "monitor"
)

In [4]:
#actions
df["action"] = np.where(
    df["reason_code"] ==
    "stale_visible_page",
    "REFRESH",
    "MONITOR"
)

In [5]:
#Build Ranked queue
queue = df.sort_values(
    "action_score",
    ascending=False
)

queue[
    [
        "content_id",
        "action_score",
        "reason_code",
        "action"
    ]
].head(20)

,content_id,action_score,reason_code,action
6653,content_5fe46e04994d,98.521830,stale_visible_page,REFRESH
17812,content_aaef01a50def,93.548392,stale_visible_page,REFRESH
26844,content_8c19996aa890,92.971339,stale_visible_page,REFRESH
21819,content_4c36c775b818,89.327748,stale_visible_page,REFRESH
29879,content_1a9e894be2e2,87.724441,stale_visible_page,REFRESH
18870,content_db5989a78dd3,80.271381,stale_visible_page,REFRESH
29400,content_2dba2b1f9536,80.102113,stale_visible_page,REFRESH
21565,content_9532f197bbc8,77.298186,stale_visible_page,REFRESH
19636,content_2cb567c3c89b,76.563973,monitor,MONITOR
13537,content_2c2606c5d176,75.937264,stale_visible_page,REFRESH


| Archetype | Action |
|------------|------------|
| Stale visible page | Refresh |
| High visibility + low CTR | Review metadata |
| Younger page with stable metrics | Monitor |
| Strong performer | Protect |

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

# Intended Use

This playbook supports content-review prioritization.

The recommendations help reviewers decide which pages may deserve attention first.

The output should be treated as decision-support.

# Limits

The recommendations do not prove that refreshing content will improve traffic.

The rankings reflect observed signals rather than causal relationships.

A high score indicates review priority, not guaranteed business impact.

## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

# Human Review Rules

Every recommendation should be reviewed by a human before action is taken.

Reviewers should inspect:

- Recent performance
- Content quality
- Seasonality
- Related content
- Business priorities

# No-Go List

The following actions should not be automated:

- Deleting content
- Publishing content changes
- Content pruning
- Strategic SEO decisions
- Client-facing recommendations

These actions require human judgment.

## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

# Monitoring and Retraining

The model or scoring system should be reviewed when:

- Content inventory changes substantially
- Search behavior changes
- Performance metrics drift
- Model quality declines
- Feature distributions change

A retraining or reassessment cycle every few months would provide an opportunity to verify that the scoring system still behaves as expected.

# Cost and Value Thinking

False Positive

A reviewer spends time inspecting a page that does not require action.

Cost:
Review effort and time.

False Negative

An important review opportunity is missed.

Cost:
Potential visibility, engagement, or content-quality improvements are delayed.

The intended goal is to reduce missed opportunities while keeping review effort reasonable.

## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [7]:
# Creating Folder
import os

os.makedirs(
    "../../work/outputs",
    exist_ok=True
)

In [8]:
#Exporting Queue
queue[
    [
        "content_id",
        "action_score",
        "reason_code",
        "action"
    ]
].to_csv(
    "../../work/outputs/action_playbook_queue.csv",
    index=False
)

print("Queue exported")

Queue exported


In [9]:
#Verifying Export
import os

os.path.exists(
    "../../work/outputs/action_playbook_queue.csv"
)

True

In [10]:
#Example Recommendations
queue[
    [
        "content_id",
        "action_score",
        "reason_code",
        "action"
    ]
].head(10)

,content_id,action_score,reason_code,action
6653,content_5fe46e04994d,98.521830,stale_visible_page,REFRESH
17812,content_aaef01a50def,93.548392,stale_visible_page,REFRESH
26844,content_8c19996aa890,92.971339,stale_visible_page,REFRESH
21819,content_4c36c775b818,89.327748,stale_visible_page,REFRESH
29879,content_1a9e894be2e2,87.724441,stale_visible_page,REFRESH
18870,content_db5989a78dd3,80.271381,stale_visible_page,REFRESH
29400,content_2dba2b1f9536,80.102113,stale_visible_page,REFRESH
21565,content_9532f197bbc8,77.298186,stale_visible_page,REFRESH
19636,content_2cb567c3c89b,76.563973,monitor,MONITOR
13537,content_2c2606c5d176,75.937264,stale_visible_page,REFRESH


# Example Recommendations

The highest-ranked pages typically exhibit:

- Meaningful visibility
- Older content age
- Potential CTR opportunity

These characteristics suggest that reviewer attention may be most valuable on these pages.

The recommendations should be interpreted as review candidates rather than guaranteed improvement opportunities.

## Self-check

Before you submit, confirm each line honestly:

- [ y] Every section above is filled — markdown thinking AND the code that backs it
- [ y] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ y] No client names, URLs, or private queries anywhere
- [ y] My claims use careful words: observed, measured, directional, decision-support
- [ y] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.